# Чат-бот на базе rugpt3medium_based_on_gpt2

## Среда

### Установка бибилиотек

In [1]:
# pip install transformers datasets torch rouge detoxify nltk bert-score tensorboard

### Стандартные бибилотеки

In [2]:
import time
import re
import json
import threading
import sys
import itertools

### Обработка данных

In [3]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

### Нейросеть

In [4]:
import torch
from torch.cuda.amp import GradScaler
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel, 
    GPT2Config,
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)

### Визуализация

In [5]:
from tqdm.notebook import tqdm

### Метрики

In [6]:
from detoxify import Detoxify
from bert_score import score

In [7]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

## Замеры времени

In [8]:
# Фрейм для замеров времени
time_metrics = pd.DataFrame(columns=["stage", "time_sec", "time_formatted"])

In [9]:
# Функция для форматирования времени
def format_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int((seconds - int(seconds)) * 100)
    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}.{milliseconds:02}"

In [10]:
# Функция для добавления времени
def add_time_metric(stage_name, time_value):
    formatted_time = format_time(time_value)
    new_row = pd.DataFrame({
        "stage": [stage_name],
        "time_sec": [round(time_value, 2)],
        "time_formatted": [formatted_time]
    })
    global time_metrics
    time_metrics = pd.concat([time_metrics, new_row], ignore_index=True)

In [11]:
start_total_time = time.time()

## Данные

### Загрузка датасета

In [12]:
dataset = load_dataset("MLNavigator/russian-retrieval")

### Очистка датасета

In [13]:
# Очистка датасета от меток SOURCE
def clean_dataset(example):
    example['q'] = re.sub(r'\s*SOURCE.*\n*.*', '', example['q'], flags=re.IGNORECASE).strip()
    example['a'] = re.sub(r'\s*SOURCE.*\n*.*', '', example['a'], flags=re.IGNORECASE).strip()
    return example

dataset = dataset.map(clean_dataset)

### Уменьшение датасета для ускорения экспериментов

In [14]:
dataset['train'] = dataset['train'].select(range(10000))

### Предобработка

In [15]:
# Разделение на train/val/test (80%/10%/10%)
split_dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_val_split = split_dataset['test'].train_test_split(test_size=0.5, seed=42)

train_data = split_dataset['train']
val_data = train_val_split['train']
test_data = train_val_split['test']

In [16]:
# Инициализация токенизатора
tokenizer = GPT2Tokenizer.from_pretrained("ai-forever/rugpt3medium_based_on_gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Установка pad_token равным eos_token для батчей

In [17]:
# Подготовка данных для модели
def preprocess_function(examples):
    inputs = [f"Вопрос: {q} Ответ: {a}" for q, a in zip(examples['q'], examples['a'])]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=64,
        return_attention_mask=True,
        return_tensors="pt"
    )
    # Метки для обуччения
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

In [18]:
# Предобработка на данных
train_data = train_data.map(preprocess_function, batched=True)
val_data = val_data.map(preprocess_function, batched=True)
test_data = test_data.map(preprocess_function, batched=True)

## Модель

### Загрузка пердобученной модели

In [19]:
# Загрузка конфигурации с отключенным кэшем
config = GPT2Config.from_pretrained("sberbank-ai/rugpt3medium_based_on_gpt2")
config.use_cache = False

In [20]:
model = GPT2LMHeadModel.from_pretrained("sberbank-ai/rugpt3medium_based_on_gpt2", config=config)
if 'lm_head.weight' not in model.state_dict():
    # Инициализация lm_head теми же весами, что и word embeddings
    model.lm_head.weight.data = model.transformer.wte.weight.data.clone()

### Оптимизация для GPU или CPU

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
device

'cuda'

In [22]:
# Очистка кэша PyTorch
torch.cuda.empty_cache()

### Настройка параметров обучения с учётом характеристик видеокарты RTX2060

In [23]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=3e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="steps",
    eval_accumulation_steps=8,
    eval_steps=200,
    save_steps=400,
    save_total_limit=2,
    fp16=True if device == "cuda" else False,
    fp16_opt_level="O1",
    gradient_checkpointing=True,
    optim="adafactor",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to= "tensorboard",
    max_grad_norm=1.0,
)

model.gradient_checkpointing_enable()

### Обучение модели

In [24]:
# Коллатор для подготовки батчей (языковое моделирование без MLM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # GPT-2 не использует masked language modeling
)

# Инициализация Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,  # data_collator вместо tokenizer
)

In [25]:
start_train_time = time.time()

In [26]:
trainer.train()
trainer.save_model("results/final_model")

Step,Training Loss,Validation Loss
200,2.778100,2.484853
400,2.241400,2.165948
600,2.036300,2.104240
800,1.730700,2.091619
1000,1.737800,2.067115
1200,1.616800,2.090983


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [27]:
end_train_time = time.time()
train_duration = end_train_time - start_train_time
print(f"Обучение заняло: {train_duration:.2f} сек")
add_time_metric("Обучение", train_duration)

Обучение заняло: 4675.40 сек


In [28]:
# Сохранения логов обучения
training_logs = trainer.state.log_history

# Сохранение в DataFrame и CSV
df_logs = pd.DataFrame([log for log in training_logs if 'loss' in log or 'eval_loss' in log])
df_logs.to_csv("rugpt_training_logs.csv", index=False)

print("Результаты обучения сохранены в rugpt_training_logs.csv")
df_logs

Результаты обучения сохранены в rugpt_training_logs.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,11.1870,2647.265381,2.400000e-07,0.04,10,NaN,NaN,NaN,NaN
1,11.2574,2885.638184,7.800000e-07,0.08,20,NaN,NaN,NaN,NaN
2,11.1171,2774.937012,1.320000e-06,0.12,30,NaN,NaN,NaN,NaN
3,9.8973,1609.801514,1.920000e-06,0.16,40,NaN,NaN,NaN,NaN
4,8.2192,865.680176,2.520000e-06,0.20,50,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
126,1.6223,58.952618,1.920000e-06,4.84,1210,NaN,NaN,NaN,NaN
127,1.5894,56.855442,1.520000e-06,4.88,1220,NaN,NaN,NaN,NaN
128,1.6206,61.963715,1.120000e-06,4.92,1230,NaN,NaN,NaN,NaN
129,1.6449,64.524002,7.200000e-07,4.96,1240,NaN,NaN,NaN,NaN


### Генерация ответа

In [29]:
# Функиця генерации ответа
def generate_answer(question):
    torch.cuda.synchronize()
    try:
        model.eval() # Переключение модели в режим инференса
        input_text = f"Вопрос: {question} Ответ:"
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            return_attention_mask=True
        ).to(device)

        start = time.time() # Время начала ответа
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],  # Передача маски внимания
                max_new_tokens=150,
                num_return_sequences=1,
                no_repeat_ngram_size=2,
                num_beams=4,
                do_sample=True,
                temperature=0.9,
                top_p=0.92,
                repetition_penalty=1.2,
                pad_token_id=tokenizer.eos_token_id
            )
        
        torch.cuda.synchronize()
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Расширенная очистка ответов
        clean_response = (
            response
            .replace("&ndash;", "-")      # Замена тире
            .replace("&mdash;", "—")      # Длинное тире
            .replace("&nbsp;", " ")       # Неразрывный пробел
            .replace("&laquo;", "«")      # Левые кавычки
            .replace("&raquo;", "»")      # Правые кавычки
        )
        
        # Удаление всех оставшихся HTML-сущностей
        clean_response = re.sub(r'&[\w#-]+;', '', clean_response)
        
        # Финальная очистка
        clean_response = (
            clean_response
            .replace("\\n", " ")          # Удаление переносов
            .replace("  ", " ")           # Двойные пробелы
            .strip()                      # Удаление пробелов по краям
        )

        # Обрезка после первой точки
        clean_response = re.split(r'[.!?]', clean_response.split('Ответ:')[-1], maxsplit=1)[0].strip()
        if not clean_response.endswith(('.', '!', '?')):
            clean_response += '.' 
        
        generated_answer = clean_response

        latency = time.time() - start # Длительность ответа
        
        return generated_answer, latency
            
    except Exception as e:
        print(f"Ошибка генерации: {e}")
        return "Не удалось сгенерировать ответ", 0.0 

# Функция генерации и сохранения ответов на тестовом датасете
def generate_answers_and_save(test_data, output_file="rugpt_generated_answers.json"):
    generated_data = []
    
    for example in tqdm(test_data, desc="Генерация ответов", unit="example", total=len(test_data)):
        q = example['q']
        true_answer = example['a']
        
        # Генерация ответа
        answer, latency = generate_answer(q)
        
        # Запись для сохранения
        generated_data.append({
            "Вопрос": q,
            "Эталонный ответ": true_answer,
            "Сгенерированный ответ": answer,
            "Время отклика (сек)": round(latency, 4)
        })
    
    # Сохранение в файл
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(generated_data, f, ensure_ascii=False, indent=2)
    print(f"Сгенерированные ответы сохранены в {output_file}")

In [30]:
start_gen_time = time.time()

In [31]:
# Запуск генерации ответов
generate_answers_and_save(test_data)

Генерация ответов:   0%|          | 0/1000 [00:00<?, ?example/s]

Сгенерированные ответы сохранены в rugpt_generated_answers.json


In [32]:
end_gen_time = time.time()
gen_duration = end_gen_time - start_gen_time
print(f"Генерация ответов заняла: {gen_duration:.2f} сек")
add_time_metric("Генерация ответов", gen_duration)

Генерация ответов заняла: 8954.68 сек


## Оценка качества модели

### Загрузка моделей

In [33]:
# Загрузка модели Detoxify для оценки токсичности ответов
detox = Detoxify('original', device=device)
def check_toxicity(text):
    return detox.predict(text)['toxicity']

### Функции для расчета метрик

In [34]:
# Перплексия для оценки предсказания текста
def calculate_perplexity(question, generated_answer):
    inputs = tokenizer(f"Вопрос: {question} Ответ: {generated_answer}", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()

In [35]:
# Ррасчёт BERTScore
def calculate_bertscore(generated, true_answer):
    P, R, F1 = score(
        [generated], 
        [true_answer], 
        lang="ru", 
        model_type="bert-base-multilingual-cased",
        verbose=False
    )
    return F1.item()

In [36]:
# Токсичность ответов
def check_toxicity(text):
    return detox.predict(text)['toxicity']

### Расчет метрик из сохраненных данных

In [37]:
# Функция расчёта метрик 
def calculate_metrics_from_file(input_file="rugpt_generated_answers.json", output_file="rugpt_metrics.csv"):
    with open(input_file, "r", encoding="utf-8") as f:
        generated_data = json.load(f)
    
    metrics = []
    
    for example in tqdm(generated_data, desc="Расчет метрик", unit="example", total=len(generated_data)):
        q = example["Вопрос"]
        true_answer = example["Эталонный ответ"]
        generated = example["Сгенерированный ответ"]
        latency = example.get("Время отклика (сек)", 0.0)
        
        # Проверка, что на входе строка
        if not isinstance(generated, str):
            generated = " ".join(generated)
        
        # Расчет метрик
        ppl = calculate_perplexity(q, generated) 
        bert_score = calculate_bertscore(generated, true_answer)
        toxicity = check_toxicity(generated)
        
        metrics.append({
            "Вопрос": q,
            "Сгенерированный ответ": generated,
            "Эталонный ответ": true_answer,
            "Perplexity": round(ppl, 2),
            "BERTScore F1": round(bert_score, 4),
            "Токсичность": round(toxicity, 4),
            "Время отклика (сек)": round(latency, 4)
        })
    
    # Создание DataFrame и сохранение
    df = pd.DataFrame(metrics)
    df.to_csv(output_file, index=False)
    print(f"Метрики сохранены в {output_file}")

    avg_latency = df["Время отклика (сек)"].mean()

    # Сводка по средним значениям
    summary = df.mean(numeric_only=True)
    print("Средние значения метрик:")
    print(summary)

### Расчёт метрик

In [38]:
start_metrics_time = time.time()

In [39]:
calculate_metrics_from_file()

Расчет метрик:   0%|          | 0/1000 [00:00<?, ?example/s]

Метрики сохранены в rugpt_metrics.csv
Средние значения метрик:
Perplexity             7.768020
BERTScore F1           0.702097
Токсичность            0.002093
Время отклика (сек)    8.950278
dtype: float64


In [40]:
end_metrics_time = time.time()
metrics_duration = end_metrics_time - start_metrics_time
print(f"Расчет метрик занял: {metrics_duration:.2f} сек")
add_time_metric("Расчёт метрик", metrics_duration)

Расчет метрик занял: 1299.73 сек


In [41]:
end_total_time = time.time()
total_duration = end_total_time - start_total_time
add_time_metric("Общее", total_duration)

In [42]:
time_metrics

,stage,time_sec,time_formatted
0,Обучение,4675.40,01:17:55.39
1,Генерация ответов,8954.68,02:29:14.67
2,Расчёт метрик,1299.73,00:21:39.72
3,Общее,14995.79,04:09:55.79


## Чат-бот

In [48]:
def chat_bot():
    print("Бот: Здравствуйте! Для выхода введите 'выход'.")
    qa_history = []
    
    # Анимация ожидания
    def loading_animation(stop_event):
        symbols = itertools.cycle(['⠇', '⠋', '⠙', '⠸', '⠴', '⠦'])
        while not stop_event.is_set():
            sys.stdout.write(f"\rБот: Думаю... {next(symbols)}")
            sys.stdout.flush()
            time.sleep(0.1)
        sys.stdout.write("\r" + " " * 40 + "\r")  # Очистка строки
    
    while True:
        original_question = input("\nВы: ").strip()
        if original_question.lower() == "выход":
            break
        
        current_question = original_question
        attempts = []
        max_attempts = 3
        success = False
        clarification_used = False  # Метка, что использовалось уточнение
        
        for attempt_num in range(1, max_attempts + 1):
            # Запуск анимации
            stop_animation = threading.Event()
            animation_thread = threading.Thread(target=loading_animation, args=(stop_animation,))
            animation_thread.start()
            
            try:
                answer, latency = generate_answer(current_question)
                clean_answer = answer.split("Ответ:")[-1].strip()
                
                # Остановка анимации
                stop_animation.set()
                animation_thread.join()
                
                # Сохранение попытки
                attempts.append({
                    "attempt": attempt_num,
                    "question": current_question,
                    "answer": clean_answer,
                    "latency": round(latency, 4),
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
                })
                
                print(f"Бот: {clean_answer} ({latency:.2f}с)")
                
                # Обратная связь
                feedback = ""
                while feedback not in ["да", "нет"]:
                    feedback = input("ℹ️ Вас устраивает ответ? (да/нет): ").lower()
                    if feedback not in ["да", "нет"]:
                        print("Введите 'да' или 'нет'")
                
                if feedback == "да":
                    success = True
                    clarification_used = attempt_num > 1
                    break
                else:
                    if attempt_num < max_attempts:
                        clarification = input("🔄 Уточните вопрос: ").strip()
                        current_question += f" ({clarification})"
        
            except Exception as e:
                stop_animation.set()
                animation_thread.join()
                print(f"\rОшибка генерации: {e}")
                answer = "Не удалось сгенерировать ответ"
                break
        
        # Создание записи
        record = {
            "original_question": original_question,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "attempts": attempts,
            "status": "clarifications" if clarification_used else "success" if success else "operator_required"
        }
        
        # Вывод статуса
        if success:
            if clarification_used:
                print(f"⚠️ Ответ с уточнениями сохранен в истории (попыток: {attempt_num})")
            else:
                print("✅ Ответ сохранен в истории успешных")
        else:
            print("❌ Бот: Передаю запрос оператору")
        
        qa_history.append(record)
        
        # Сохранение в файл
        with open("rugpt_qa_full_history.json", "w", encoding="utf-8") as f:
            json.dump(qa_history, f, ensure_ascii=False, indent=2)

    print("Бот: До свидания!")

In [49]:
# Запуск чат-бота
if __name__ == "__main__":
    chat_bot()

Бот: Здравствуйте! Для выхода введите 'выход'.




Вы:  кто президент РФ с 2012 года?


Бот: В 2012 году Владимир Путин был избран президентом Российской Федерации по итогам второго тура президентских выборов. (41.73с)


ℹ️ Вас устраивает ответ? (да/нет):  да


✅ Ответ сохранен в истории успешных


...

Бот: Первая Мировая война началась 1 (14) июля 1914 года, когда германские войска вторглись в Австро-Венгрию и Румынию. (41.08с)


ℹ️ Вас устраивает ответ? (да/нет):  да


✅ Ответ сохранен в истории успешных



Вы:  выход


Бот: До свидания!
